# Bayonne with a car-free old town

The morning commute across the Bayonne–Anglet–Biarritz area and its hinterland: census 2022
home–work flows, OpenStreetMap roads, day-to-day route and mode choice in SPARQL on a
knowledge graph ([skabm](https://github.com/remiadon/skabm)). Not calibrated on traffic
counts yet: read the differences, not the levels.

Sources, all open: roads from OpenStreetMap through Overpass (ODbL); communes, populations
and outlines from geo.api.gouv.fr; home-to-work flows by commune pair and mode (census 2022)
from INSEE, as republished on data.gouv.fr.

In [ ]:
# uv run --extra voila voila notebooks   (or open this notebook in Jupyter)
import io
import json
import math
import time
from pathlib import Path

import httpx
import ipywidgets as w
import matplotlib

matplotlib.use("Agg")  # figures become PNGs for an Image widget
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pyarrow as pa
import sympy as sp
from IPython.display import Markdown, display
from lonboard import Map, PathLayer
from lonboard.basemap import CartoStyle, MaplibreBasemap
from maplib import Model
from matplotlib.path import Path as Polygon
from sklearn.cluster import KMeans

from skabm import template
from skabm.behaviour import defaults
from skabm.behaviour.traffic import (
    MODES,
    TRAFFIC_UDFS,
    Link,
    area_membership,
    bayonne_link_time,
    bayonne_rules,
    bpr_alpha,
    bpr_beta,
    pedestrianize,
    routes,
)
from skabm.player import Player
from skabm.simulation import RDFSimulator
from skabm.sparql import _PREFIXES

# The world: OSM roads, 2022 census commuting flows, and the populations
# skabm.behaviour.traffic expects.  Specific to Bayonne, like the page below.

CACHE = Path("data") / "bayonne"  # notebooks/data/bayonne: delete a file there to refetch it
# Every street from Biarritz to Tarnos, the coast to the A63/A64 junction; and the
# main roads of the whole commuting basin, Hendaye to Dax, the coast to Saint-Palais,
# so the inland arrives by the A64, the D932 and the D936 rather than cross-country.
SOUTH, WEST, NORTH, EAST = 43.425, -1.600, 43.565, -1.385
REGION = (43.15, -1.80, 43.80, -0.95)
FARTHEST = 80_000.0  # metres, crow-fly: a census "usual workplace" farther is rarely daily
LON0, LAT0 = -1.475, 43.49  # local metric projection origin
FLOWS = "39df0b9c-dd76-44ce-8a15-8c00d8a211c1"  # data.gouv.fr resource: flows by mode

# Drawn on OSM, not taken from it (OSM has the quarters as points only).  Grand
# Bayonne: west of the Nive, south of the Adour, north of the ramparts' road, east of
# Allées Paulmy, which stays outside it.
# The Nive bridges cross the east edge, so closing the area closes them.
AREAS = {
    # east edge: mid-river through the Mayou, Marengo, Pannecau and Génie bridges
    "Grand Bayonne": "POLYGON((-1.4781 43.4945, -1.4762 43.4941, -1.4742 43.4936, "
    "-1.4741 43.4923, -1.4744 43.4910, -1.4746 43.4897, -1.4749 43.4879, "
    "-1.4762 43.4876, -1.4801 43.4884, -1.4798 43.4914, -1.4791 43.4931, "
    "-1.4781 43.4945))",
    "Petit Bayonne": "POLYGON((-1.4742 43.4936, -1.4728 43.4935, -1.4700 43.4922, "
    "-1.4686 43.4910, -1.4690 43.4878, -1.4722 43.4868, -1.4749 43.4879, "
    "-1.4746 43.4897, -1.4744 43.4910, -1.4741 43.4923, -1.4742 43.4936))",
}

_MAJOR = "motorway|motorway_link|trunk|trunk_link|primary|primary_link|secondary|secondary_link"
_HIGHWAYS = _MAJOR + "|tertiary|tertiary_link|unclassified|residential|living_street|busway|service|pedestrian"
# km/h and veh/h per lane when OSM says nothing (French defaults; MATSim's OSM reader
# for capacities).
_DEFAULTS = {
    "motorway": (110, 2000), "trunk": (90, 1800), "primary": (50, 1500),
    "secondary": (50, 1000), "tertiary": (50, 800), "unclassified": (40, 600),
    "residential": (30, 500), "living_street": (20, 300), "pedestrian": (10, 300),
    # only bus-designated service roads survive _access: bus lanes, and a lane if reopened
    "busway": (30, 1000), "service": (30, 1000),
}
_MAXSPEED = {"FR:urban": 50, "FR:rural": 80, "FR:zone30": 30, "FR:motorway": 130, "walk": 6}
# Census "mode_transport" to model mode; working from home is no trip.
_MODE = {"Voiture": "car", "Deux-roues motorisé": "car", "Transports en commun": "bus",
         "Vélo": "bike", "Marche": "walk"}


def _cached(name: str, fetch) -> object:
    path = CACHE / name
    if not path.exists():
        CACHE.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(fetch()))
    return json.loads(path.read_text())


def _get(url: str, **params) -> object:
    response = httpx.get(url, params=params, timeout=180,
                         headers={"User-Agent": "skabm-research (github.com/remiadon/skabm)"})
    response.raise_for_status()
    return response.json()


def metres(lon, lat):
    """Local equirectangular projection around Bayonne — metres east and north."""
    return ((lon - LON0) * 111_320 * math.cos(math.radians(LAT0)), (lat - LAT0) * 110_574)

In [ ]:
# Roads

def osm() -> dict:
    query = f"""[out:json][timeout:170];
    (way["highway"~"^({_HIGHWAYS})$"]({SOUTH},{WEST},{NORTH},{EAST});
     way["highway"~"^({_MAJOR})$"]{REGION};);
    out body; >; out skel qt;"""
    return _cached("osm.json", lambda: _get("https://overpass-api.de/api/interpreter", data=query))


def _speed(tags: dict, kind: str) -> float:
    raw = tags.get("maxspeed", "")
    if raw in _MAXSPEED:
        return _MAXSPEED[raw]
    number = raw.split()[0] if raw else ""
    if number.isdigit():
        return float(number) * (1.609 if "mph" in raw else 1)
    return _DEFAULTS[kind][0]


def _access(tags: dict, kind: str) -> tuple[bool, bool, bool]:
    """(cars may use it, buses may, it is a bus lane or bus-only)."""
    motor = tags.get("motor_vehicle", tags.get("motorcar", tags.get("vehicle")))
    closed = kind in ("busway", "pedestrian") or tags.get("access") in ("no", "private")
    car = motor in ("yes", "destination", "designated") if (closed or motor) else True
    if kind == "service" and "designated" not in (tags.get("bus"), tags.get("psv")):
        return False, False, False  # driveways and car parks are not the network
    bus = car or tags.get("bus") in ("yes", "designated") or tags.get("psv") in ("yes", "designated")
    lane = "designated" in tags.get("bus:lanes", "") or tags.get("busway") in ("lane", "opposite_lane")
    return car, bus, (bus and not car) or lane


def network() -> tuple[pl.DataFrame, pl.DataFrame]:
    """Directed links between intersections, and the nodes they join (metres)."""
    data = osm()["elements"]
    where = {e["id"]: (e["lon"], e["lat"]) for e in data if e["type"] == "node"}
    ways = [e for e in data if e["type"] == "way"]
    seen: dict = {}
    for way in ways:
        for i, node in enumerate(way["nodes"]):
            seen[node] = seen.get(node, 0) + (2 if i in (0, len(way["nodes"]) - 1) else 1)
    rows = []
    for way in ways:
        tags = way["tags"]
        kind = tags["highway"].removesuffix("_link")
        car, bus, busway = _access(tags, kind)
        if not bus:
            continue
        oneway = tags.get("oneway") in ("yes", "true", "1") or (
            kind == "motorway" or tags.get("junction") == "roundabout"
        ) and tags.get("oneway") != "no"
        reverse = tags.get("oneway") == "-1"
        lanes = float(tags.get("lanes", "0").split(";")[0] or 0) or (2 if kind == "motorway" else 1)
        lanes = lanes if oneway or reverse else max(1.0, lanes / 2)
        speed, per_lane = _speed(tags, kind), _DEFAULTS[kind][1]
        cut = [0] + [i for i, n in enumerate(way["nodes"][1:-1], 1) if seen[n] > 1] + [len(way["nodes"]) - 1]
        for a, b in zip(cut, cut[1:]):
            chain = way["nodes"][a : b + 1]
            xy = [metres(*where[n]) for n in chain]
            length = sum(math.dist(p, q) for p, q in zip(xy, xy[1:]))
            line = ", ".join(f"{where[n][0]:.6f} {where[n][1]:.6f}" for n in chain)
            for src, dst, points in (
                [] if reverse else [(chain[0], chain[-1], line)]
            ) + ([] if oneway else [(chain[-1], chain[0], ", ".join(reversed(line.split(", "))))]):
                rows.append((f"w{way['id']}_{a}_{src}", str(src), str(dst), max(length, 1.0),
                             max(length, 1.0) / (speed / 3.6), lanes * per_lane, float(car),
                             float(busway), tags.get("name"), kind, f"LINESTRING({points})"))
    links = pl.DataFrame(
        rows, orient="row",
        schema=["id", "src", "dst", "length", "t0", "capacity", "car", "busway", "name", "kind", "geometry"],
    ).unique("id", maintain_order=True)
    used = pl.concat([links["src"], links["dst"]]).unique(maintain_order=True)
    nodes = pl.DataFrame(
        [(str(n), *where[n], *metres(*where[n])) for n in map(int, used)],
        schema=["id", "lon", "lat", "x", "y"], orient="row",
    )
    return links, nodes

In [ ]:
# People: communes, flows, zones

def communes() -> pl.DataFrame:
    """Every commune of the Pyrénées-Atlantiques and the Landes: code, name, population, centre."""
    found = []
    for department in ("64", "40"):
        found += _cached(
            f"communes_{department}.json",
            lambda d=department: _get("https://geo.api.gouv.fr/communes", codeDepartement=d,
                                      fields="code,nom,population,centre"),
        )
    frame = pl.DataFrame(
        [(c["code"], c["nom"], float(c.get("population") or 0), *c["centre"]["coordinates"]) for c in found],
        schema=["code", "name", "population", "lon", "lat"], orient="row",
    )
    x, y = metres(frame["lon"], frame["lat"])
    inside = frame["lon"].is_between(WEST, EAST) & frame["lat"].is_between(SOUTH, NORTH)
    return frame.with_columns(x=x, y=y, internal=inside)


def outline(code: str) -> np.ndarray:
    shape = _cached(f"contour_{code}.json", lambda: _get(
        f"https://geo.api.gouv.fr/communes/{code}", fields="contour", format="json"))["contour"]
    parts = [shape["coordinates"]] if shape["type"] == "Polygon" else shape["coordinates"]
    return np.array(max((part[0] for part in parts), key=len))  # outer ring of the largest part


def flows(internal: list[str]) -> pl.DataFrame:
    """Workers by (home, work, mode), every pair with an internal end."""
    def fetch():
        rows = []
        url = f"https://tabular-api.data.gouv.fr/api/resources/{FLOWS}/data/"
        for side in ("code_com_1__in", "code_com_2__in"):
            page, total = 1, 1
            while (page - 1) * 200 < total:  # the API offers a `next` past the last page
                body = _get(url, **{side: ",".join(internal), "page_size": 200, "page": page})
                rows += body["data"]
                page, total = page + 1, body["meta"]["total"]
        return rows
    raw = pl.DataFrame(_cached("flows.json", fetch))
    return (
        raw.unique("__id")
        .filter(pl.col("mode_transport").is_in(list(_MODE)))
        .select(home="code_com_1", work="code_com_2",
                mode=pl.col("mode_transport").replace_strict(_MODE), workers="valeur")
        .group_by("home", "work", "mode").agg(pl.col("workers").sum())
        .sort("home", "work", "mode")  # the sample is drawn row by row: fix the rows' order
    )


def zones(towns: pl.DataFrame, nodes: pl.DataFrame, per_zone: float = 1500.0) -> pl.DataFrame:
    """Sub-zones of the internal communes; every other commune is one zone at its centre.

    An internal commune is cut into ~one zone per ``per_zone`` inhabitants by k-means
    over its street nodes (few inhabitants per zone keeps any one street from
    loading a whole neighbourhood's trips), each zone weighted by its node count — street density
    standing in for where people live and work — and reached on foot.  An external
    commune is driven from, at ``DRIVE``, to its nearest road of the regional network.

    ponytail: residents and jobs share one weight; POIs or SIRENE establishments
    would separate them when destination choice starts to matter.
    """
    points = nodes.select("x", "y").to_numpy()
    parts, taken = [], np.zeros(len(points), dtype=bool)
    for code, population in towns.filter("internal").select("code", "population").iter_rows():
        ring = np.array([metres(lon, lat) for lon, lat in outline(code)])
        mine = Polygon(ring).contains_points(points) & ~taken
        taken |= mine
        if mine.sum() == 0:
            continue
        k = int(min(max(round(population / per_zone), 1), 40, mine.sum()))
        fit = KMeans(k, n_init=4, random_state=0).fit(points[mine])
        weight = np.bincount(fit.labels_, minlength=k)
        parts.append(pl.DataFrame({
            "id": [f"z{code}_{i}" for i in range(k)], "code": code,
            "x": fit.cluster_centers_[:, 0], "y": fit.cluster_centers_[:, 1],
            "weight": weight / weight.sum(), "speed": WALK,
        }))
    parts.append(towns.filter(~pl.col("internal")).select(
        id=pl.format("z{}", "code"), code="code", x="x", y="y", weight=pl.lit(1.0), speed=pl.lit(DRIVE),
    ))
    return pl.concat(parts)


def commuters(od: pl.DataFrame, zone: pl.DataFrame, sample: float, seed: int = 0) -> pl.DataFrame:
    """A *sample* of the workers as agents, each placed in a zone of its home and work communes.

    Counts are rounded stochastically (a flow of 3.4 workers at 10% is one agent with
    probability 0.34), so small flows survive in expectation; the zone within a
    commune is drawn by weight.
    """
    rng = np.random.default_rng(seed)
    expected = od["workers"].to_numpy() * sample
    count = np.floor(expected + rng.random(len(expected))).astype(int)
    agents = od.with_columns(n=pl.Series(count)).filter(pl.col("n") > 0).select(
        "home", "work", "mode", pl.int_ranges("n").alias("k")
    ).explode("k")

    def place(codes: pl.Series) -> list[str]:
        out = []
        members = {c: g for (c,), g in zone.group_by("code")}
        for code in codes:
            options = members.get(code)
            out.append(None if options is None else rng.choice(options["id"].to_numpy(), p=options["weight"].to_numpy()))
        return out

    at = zone.select("id", "x", "y")
    return (
        agents.with_columns(origin=pl.Series(place(agents["home"])),
                            destination=pl.Series(place(agents["work"])))
        .drop_nulls()
        .join(at.rename({"id": "origin", "x": "xo", "y": "yo"}), on="origin")
        .join(at.rename({"id": "destination", "x": "xd", "y": "yd"}), on="destination")
        .filter(((pl.col("xo") - pl.col("xd")) ** 2 + (pl.col("yo") - pl.col("yd")) ** 2).sqrt() < FARTHEST)
        .select("home", "work", "mode", "origin", "destination",
                od=pl.format("{}_{}", "origin", "destination"), weight=pl.lit(1 / sample),
                id=pl.format("c{}", pl.int_range(pl.len())))
    )

In [ ]:
# The world

WALK, BIKE, DRIVE = 1.3, 4.2, 11.0  # m/s: walking, cycling, driving local roads
REACH = {"walk": 4_000.0, "bike": 15_000.0}  # metres crow-fly between zones: beyond, a census artefact
BUS_WAIT = 480.0  # seconds: half a typical headway


def choice_set(links, nodes, zone, ods, modes: pl.DataFrame, cost: str = "t0"):
    """Routes and via for every OD x mode in *modes* (``od``, ``mode``); car on the links
    that admit cars, bus on the links that admit buses, bike and walk crow-fly."""
    found, via = [], []
    for mode, allowed in (("car", pl.col("car") > 0), ("bus", pl.lit(True))):
        wanted = ods.join(modes.filter(pl.col("mode") == mode), on="od", how="semi")
        if wanted.height:
            r, v = routes(links.filter(allowed), nodes, zone, wanted, MODES[mode], cost)
            found.append(r.with_columns(extra=pl.col("extra") + (BUS_WAIT if mode == "bus" else 0.0)))
            via.append(v)
    ends = ods.join(zone.select(origin="id", xo="x", yo="y"), on="origin").join(
        zone.select(destination="id", xd="x", yd="y"), on="destination")
    crow = ((pl.col("xo") - pl.col("xd")) ** 2 + (pl.col("yo") - pl.col("yd")) ** 2).sqrt().clip(500) * 1.3
    for mode, speed in (("bike", BIKE), ("walk", WALK)):
        wanted = ends.join(modes.filter(pl.col("mode") == mode), on="od", how="semi")
        found.append(wanted.select(
            id=pl.format("{}_m{}", "od", pl.lit(int(MODES[mode]))), od="od",
            rank=pl.lit(float(MODES[mode])), extra=crow / speed, time=crow / speed,
            mode=pl.lit(MODES[mode]),
        ))
    routed = pl.concat(found, how="vertical_relaxed").select(
        "id", "od", "rank", "mode", "extra", "time",
        option=pl.format("{}_{}", "od", pl.col("mode").cast(pl.Int8)))
    return routed, pl.concat(via)


def frames(sample: float = 0.1, seed: int = 0) -> dict:
    """Everything the world is made of, as polars frames — cacheable, unlike a Model."""
    links, nodes = network()
    towns = communes()
    od = flows(towns.filter("internal")["code"].to_list())
    zone = zones(towns, nodes)
    people = commuters(od, zone, sample, seed)
    ods = people.select("od", "origin", "destination", "home", "work").unique(maintain_order=True)
    # the pivot's base is the census share of the commune pair, not of the few agents
    # a zone pair happens to draw: every mode the pair uses is open to every agent on it —
    # within reach.  The census has Capbreton residents walking to Bayonne; the pivot
    # would keep that 25 km walk open forever, so out-of-reach modes go, and so do the
    # agents drawn onto them.
    at = zone.select("id", "x", "y")
    reach = ods.join(at.rename({"id": "origin", "x": "xo", "y": "yo"}), on="origin").join(
        at.rename({"id": "destination", "x": "xd", "y": "yd"}), on="destination"
    ).select("od", "home", "work", metres=((pl.col("xo") - pl.col("xd")) ** 2 + (pl.col("yo") - pl.col("yd")) ** 2).sqrt())
    census = (
        reach.join(od, on=["home", "work"])
        .filter(pl.col("metres") <= pl.col("mode").replace_strict(REACH, default=float("inf")))
        .select("od", "mode", share0=pl.col("workers") / pl.col("workers").sum().over("od"))
    )
    people = people.join(census.select("od", "mode"), on=["od", "mode"], how="semi")
    found, via = choice_set(links, nodes, zone, ods, census.select("od", "mode"))
    # everyone starts on a route of the mode the census gives them
    first = found.sort("rank").unique(["od", "mode"], keep="first")
    code = pl.col("mode").replace_strict(MODES)
    agents = people.with_columns(mode=code).join(first.select("od", "mode", route="id"), on=["od", "mode"])
    options = census.with_columns(mode=code).with_columns(
        id=pl.format("{}_{}", "od", pl.col("mode").cast(pl.Int8)))
    return {"links": links, "nodes": nodes, "zones": zone, "ods": ods, "census": census,
            "commuters": agents, "options": options, "routes": found, "via": via,
            "communes": towns, "flows": od}


def world(parts: dict) -> Model:
    """A fresh maplib Model of *parts* — a fit advances the one it is given."""
    model = Model()  # every frame in id order: the draws follow the rows, so a seed is a seed
    model.map(template.link, parts["links"].drop("kind").with_columns(time=pl.col("t0")).sort("id").with_iri("src", "dst"))
    areas = pl.DataFrame({"id": ["grand", "petit"], "name": list(AREAS), "geometry": list(AREAS.values())})
    model.map(template.area, areas.with_iri())
    links = parts["links"].sort("id")
    model.map(template.links("area"), area_membership(links, areas).sort("id", "area").with_iri("area"))
    model.map(template.option, parts["options"].sort("id").with_iri())
    model.map(template.route, parts["routes"].sort("id").with_iri("option"))
    model.map(template.via, parts["via"].sort("id", "via").with_iri("via"))
    model.map(template.commuter, parts["commuters"].select("id", "od", "route", "weight").sort("id").with_iri("route"))
    return model


def reroute(model: Model, parts: dict) -> int:
    """Car routes against the network as the graph has it now, on today's times.

    The column-generation step: after an intervention (or congestion) the choice set
    gains each OD's current shortest path, mapped straight into the live model — a
    path already known maps to the same route and adds nothing.  Returns how many
    routes the model has now.
    """
    state = model.query(
        _PREFIXES + "SELECT ?l ?car ?time WHERE { ?l a ex:Link ; def:car ?car ; def:time ?time }"
    ).select(id=pl.col("l").str.slice(len("<http://example.net/skabm#")).str.strip_suffix(">"),
             car="car", time="time")
    links = parts["links"].drop("car").join(state, on="id")
    cars = parts["census"].filter(pl.col("mode") == "car").select("od", "mode")
    found, via = choice_set(links, parts["nodes"], parts["zones"], parts["ods"], cars, cost="time")
    model.map(template.route, found.sort("id").with_iri("option"))
    model.map(template.via, via.sort("id", "via").with_iri("via"))
    return model.query(_PREFIXES + "SELECT (COUNT(?r) AS ?n) WHERE { ?r a ex:Route }")["n"][0]

In [ ]:
# Scenarios and the protocol

# Areas closed to cars, streets (re)opened to them.
# What a scenario closes to cars: (areas, streets).  The corridor is the through route
# across the old town, Saint-Esprit to Allées Marines; Pont Mayou is the Nive crossing
# between the two places the corridor names, so it goes with them.
CORRIDOR = ("Place du Réduit", "Pont Mayou", "Quai Amiral Lespes", "Place de la Liberté",
            "Rue Bernède", "Place Général de Gaulle", "Avenue Léon Bonnat")
SCENARIOS = {
    "Today": ((), ()),
    "Old-town axis closed to cars": ((), CORRIDOR),
    "Fêtes perimeter: Grand + Petit Bayonne car-free": (("Grand Bayonne", "Petit Bayonne"), ()),
}
# Commuters on the road relative to a school-term weekday, i.e. `demand`.  Assumptions
# standing in for counts, not measurements: the knob a seasonal count calibrates.
PERIODS = {
    "School-term weekday": 1.0,
    "School holidays": 0.85,
    "July (fêtes de Bayonne)": 0.85,
    "August": 0.7,
}
PARAMS = {"peak_factor": 0.7}
KNOBS = {**defaults(), **PARAMS}  # every value a run reads: the rules' defaults, Bayonne's on top
# Cars on the road in each half hour, as a share of the 8 o'clock peak the rules settle.
# An assumption, and the first thing a permanent counter's own hourly profile replaces.
MORNING = {"06:00": 0.22, "06:30": 0.35, "07:00": 0.55, "07:30": 0.78, "08:00": 1.0}


WARMUP, DAYS, EVERY = 15, 15, 5  # days settling the base, days lived after it, days between re-routes


def simulate(parts: dict, scenario: str, period: str, params: dict | None = None,
             seed: int = 0, warmup: int = WARMUP, days: int = DAYS, every: int = EVERY):
    """Warm up on the census, then live *days* under *scenario* and *period*.

    Yields ``(phase, frame, sim)`` per day, *frame* every agent after it.  The warm-up anchors every OD's mode
    shares on the census at school-term demand while routes equilibrate, re-routing
    on congested times every *every* days; then the anchor is dropped, demand
    becomes the period's, the scenario's streets close and open, and the choice set
    is re-routed once more — for "Today" too, so a scenario differs from it by the
    policy alone.  The draws line up across scenarios (one per commuter per day, one
    seed), so two runs are paired: common random numbers.
    """
    params = {**PARAMS, **(params or {})}
    areas, streets = SCENARIOS[scenario]
    model = world(parts)
    sim = RDFSimulator(rules=bayonne_rules,
                       params={**params, "anchor": 1.0, "demand": 1.0}, udfs=TRAFFIC_UDFS,
                       n_periods=every, random_seed=seed)
    for segment in range(warmup // every):
        for frame in sim.fit_iter(model if segment == 0 else None):
            yield "warm-up", frame, sim
        reroute(sim.model_, parts)
        sim.set_params(warm_start=True, random_seed=None)  # one draw stream, not restarts
    unknown = set(streets) - set(parts["links"]["name"].drop_nulls())
    if unknown:
        raise ValueError(f"no link is named {sorted(unknown)}: nothing would close")
    if areas or streets:
        sim.model_.update(pedestrianize(areas, streets))
    reroute(sim.model_, parts)
    sim.set_params(n_periods=days, params={**params, "anchor": 0.0, "demand": PERIODS[period]})
    for frame in sim.fit_iter():
        yield scenario, frame, sim


def passes(scenario: str, period: str, sample: float, knobs: dict):
    """A whole run, one settled pass at a time, in plain frames: the pass's commuter
    means, its vehicle-km in Grand Bayonne, and its link flows.  Nothing of the graph
    outlives the run."""
    parts = frames(sample=sample)
    for phase, frame, sim in simulate(parts, scenario, period, knobs):
        commuters = frame.filter(pl.col("class") == "Commuter")
        day = commuters.select(pl.col("time", "car", "bus").mean()).row(0, named=True)
        yield {"day": frame["t"][0], "phase": phase, **day, "vkt": area_state(sim)["Grand Bayonne"],
               "flows": link_state(sim).select("id", "flow")}


def paths(links: pl.DataFrame) -> pl.Series:
    """Each link's WKT as a list of [lon, lat] — what a map layer draws."""
    return links["geometry"].str.extract(r"\((.*)\)").str.split(", ").list.eval(
        pl.element().str.split(" ").cast(pl.List(pl.Float64)))


def area_state(sim) -> dict:
    """Vehicle-km per peak hour inside each Area, by name."""
    rows = sim.model_.query(_PREFIXES + "SELECT ?name ?vkt WHERE { ?a a ex:Area ; def:name ?name ; def:vkt ?vkt }")
    return dict(rows.iter_rows())


def link_state(sim) -> pl.DataFrame:
    """Every link's flow, time and access now, joined back to its name and geometry."""
    return sim.model_.query(
        _PREFIXES + "SELECT ?l ?flow ?time ?car WHERE { ?l a ex:Link ; def:flow ?flow ; def:time ?time ; def:car ?car }"
    ).select(id=pl.col("l").str.slice(len("<http://example.net/skabm#")).str.strip_suffix(">"),
             flow="flow", time="time", car="car")


# The delay is the BPR of ``traffic.bayonne_link_time``, evaluated from the rule itself
delay = sp.lambdify([Link.t0, Link.flow, Link.capacity, bpr_alpha, bpr_beta], bayonne_link_time[Link.time])


def at_hour(flows: pl.DataFrame, links: pl.DataFrame, share: float, params: dict | None = None) -> pl.DataFrame:
    """The settled peak scaled down to one half hour, and the delay that follows from it.

    A replay, not a simulation: the rules settle the peak, and the hours before it are
    that loading scaled by ``MORNING`` — so the flows fall linearly and the delay does
    not, which is the jam forming.  No queue carries over from one half hour to the next.
    """
    knobs = {**KNOBS, **(params or {})}
    frame = flows.join(links.select("id", "capacity", "t0"), on="id").with_columns(flow=pl.col("flow") * share)
    time = delay(frame["t0"].to_numpy(), frame["flow"].to_numpy(), frame["capacity"].to_numpy(),
                 knobs["bpr_alpha"], knobs["bpr_beta"])
    return frame.with_columns(ratio=pl.col("flow") / pl.col("capacity"), time=pl.Series(time))

In [ ]:
# The page.  dataviz reference palette: the blue sequential ramp, the blue <-> red diverging pair
RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
LESS, MIDDLE, MORE = "#2a78d6", "#f0efec", "#e34948"
GOOD, BAD, MUTED = "#1a7f37", "#c62828", "#6b6a65"
# The three mornings worth looking at.  Everything between them is the solver settling
# the assignment, not days of Bayonne, so it stays under the hood.
STATES = {"Before the closure": "before", "The first morning after": "first", "Once drivers have adjusted": "last"}
SHOWN = {"Mean commute": ("time", 1 / 60, "{:.1f} min", "inverse"),
         "Commuting by car": ("car", 100, "{:.1f}%", "inverse"),
         "By public transport": ("bus", 100, "{:.1f}%", "normal"),
         "Cars in Grand Bayonne": ("vkt", 1, "{:,.0f} veh·km/h", "inverse")}

parts = frames(sample=0.1)  # roads, zones and census flows; a run draws its own sample
links = parts["links"].with_columns(pl.col("name").fill_null("unnamed road"))
order = links.select("id")  # the layer's row order: every colour and width follows it


def road_layer() -> PathLayer:
    """Every road link as one deck.gl path layer: the geometry goes to the browser once,
    then only colours and widths change."""
    crs = {b"ARROW:extension:name": b"geoarrow.linestring",
           b"ARROW:extension:metadata": json.dumps({"crs": "OGC:CRS84"}).encode()}
    geometry = pa.array(links.select(path=paths(links))["path"].to_list(), type=pa.list_(pa.list_(pa.float64(), 2)))
    table = pa.Table.from_arrays(
        [geometry, links["name"].to_arrow(), links["capacity"].to_arrow()],
        schema=pa.schema([pa.field("geometry", geometry.type, metadata=crs),
                          pa.field("name", pa.large_string()), pa.field("capacity (veh/h)", pa.float64())]),
    )
    return PathLayer(table, get_color=[120, 120, 118], get_width=0.6 + 2 * np.clip(links["capacity"].to_numpy() / 2000, 0, 1),
                     width_units="pixels", width_min_pixels=1, cap_rounded=True, pickable=True, auto_highlight=True)


def colour(t: np.ndarray, stops: list[str], shown: np.ndarray) -> np.ndarray:
    """RGBA, piecewise-linear along *stops* for t in [0, 1]; a link not *shown* is transparent."""
    table = np.array([[int(h[i : i + 2], 16) for i in (1, 3, 5)] for h in stops], dtype=float)
    at = np.linspace(0, 1, len(stops))
    rgb = np.stack([np.interp(np.clip(t, 0, 1), at, table[:, k]) for k in range(3)], 1)
    return np.column_stack([rgb, np.where(shown, 255, 0)]).round().astype(np.uint8)


def peak_at(flows: pl.DataFrame, share: float, knobs: dict) -> pl.DataFrame:
    """Every link, in the layer's order, at *share* of the settled peak (``world.at_hour``)."""
    return order.join(at_hour(flows, links, share, knobs), on="id", how="left", maintain_order="left").fill_null(0)


def jam(frame: pl.DataFrame) -> int:
    """Congestion as it stands: volume over capacity, light to dark; under 50 veh/h hidden.
    Returns how many roads are over capacity."""
    flow = frame["flow"].to_numpy()
    layer.get_color = colour(frame["ratio"].to_numpy() / 1.25, RAMP, flow >= 50)
    layer.get_width = (1 + 5 * np.clip(flow / 3000, 0, 1)).astype(np.float32)
    return int((flow > frame["capacity"].to_numpy()).sum())


def moved(frame: pl.DataFrame, was: np.ndarray) -> int:
    """What the closure moved: peak flows against before it; under 25 veh/h of change hidden."""
    delta = frame["flow"].to_numpy() - was
    layer.get_color = colour((delta / 1500 + 1) / 2, [LESS, MIDDLE, MORE], np.abs(delta) >= 25)
    layer.get_width = (1.5 + 6 * np.clip(np.abs(delta) / 1500, 0, 1)).astype(np.float32)
    return int((frame["flow"].to_numpy() > frame["capacity"].to_numpy()).sum())


def card(label: str, value: str, note: str = "", tone: str = MUTED) -> str:
    return (f"<div style='border:1px solid #e6e5e0;border-radius:8px;padding:6px 12px;min-width:150px'>"
            f"<div style='font-size:12px;color:{MUTED}'>{label}</div><div style='font-size:22px'>{value}</div>"
            f"<div style='font-size:12px;color:{tone}'>{note}&nbsp;</div></div>")


def row(*cards: str) -> str:
    return "<div style='display:flex;gap:10px;flex-wrap:wrap'>" + "".join(cards) + "</div>"


def figures(now: dict, base: dict | None) -> None:
    """A pass's headline numbers, against *base* (none before the closure)."""
    cards = []
    for label, (column, scale, form, sense) in SHOWN.items():
        value, note, tone = now[column] * scale, "", MUTED
        if base is not None:
            gap = value - base[column] * scale
            note = ("+" if gap > 0 else "") + form.format(gap)
            tone = MUTED if gap == 0 else BAD if (gap > 0) == (sense == "inverse") else GOOD
        cards.append(card(label, form.format(value), note, tone))
    numbers.value = row(*cards)


def dial(label: str, value: str, note: str, over: int) -> None:
    dials.value = row(card(label, value, note), card("Roads over capacity", f"{over:,}"))


def settling(rows: list) -> None:
    """Mean commute per pass: the solver settling, then the closure's shock and recovery."""
    fig, ax = plt.subplots(figsize=(9, 2.2), layout="constrained")
    ax.plot([r["day"] for r in rows], [r["time"] / 60 for r in rows], color=MORE, linewidth=2)
    ax.axvline(WARMUP + 0.5, color="#b4b2a9", linewidth=1, linestyle="--")
    ax.set_xlim(1, WARMUP + DAYS)
    ax.set_xlabel("pass")
    ax.set_title("Mean commute (min)", loc="left", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=90)
    plt.close(fig)
    curve.value = buffer.getvalue()
    settled.layout.display = None


def streets(now: dict, base: dict) -> None:
    """The twelve named streets whose peak traffic changes most on the morning *now*."""
    change = (now["flows"].join(base["flows"].rename({"flow": "was"}), on="id").join(links.select("id", "name"), on="id")
              .with_columns(delta=pl.col("flow") - pl.col("was")).filter(pl.col("name") != "unnamed road")
              .sort(pl.col("delta").abs(), descending=True).unique("name", maintain_order=True).head(12))
    cells = "".join(f"<tr><td>{r['name']}</td><td>{r['was']:,.0f}</td><td>{r['flow']:,.0f}</td><td>{r['delta']:+,.0f}</td></tr>"
                    for r in change.iter_rows(named=True))
    table.value = (f"<b>Streets whose peak traffic changes most</b>: {morning.value.lower()}, against before the closure"
                   "<table style='border-collapse:collapse;margin-top:6px'><tr><th>Street</th><th>Before (veh/h)</th>"
                   f"<th>That day (veh/h)</th><th>Change</th></tr>{cells}</table>"
                   "<style>table td, table th {padding: 2px 12px; text-align: right} table td:first-child, "
                   "table th:first-child {text-align: left}</style>")


wide = {"description_width": "11em"}
scenario = w.Dropdown(options=list(SCENARIOS), index=1, description="Scenario", style=wide)
period = w.Dropdown(options=list(PERIODS), description="Period of the year", style=wide,
                    tooltip="Commuting volume relative to a school-term weekday.")
mu = w.FloatSlider(KNOBS["mu"], min=0.05, max=1.0, step=0.01, description="Mode sensitivity μ", style=wide,
                   tooltip="How strongly mode shares answer a change in travel time (nest scale ratio).")
replan = w.FloatSlider(KNOBS["replan"], min=0.02, max=0.5, step=0.01, description="Reconsidering each day", style=wide)
peak = w.FloatSlider(KNOBS["peak_factor"], min=0.3, max=1.2, step=0.05, description="Peak vehicles per car commuter",
                     style=wide, tooltip="Share of commutes in the peak hour, times the uplift for non-commute traffic.")
sample = w.ToggleButtons(options=[("5%", 0.05), ("10%", 0.1), ("20%", 0.2)], value=0.1, description="Agent sample",
                         style={**wide, "button_width": "4em"})
for control in (scenario, period, mu, replan, peak):
    control.layout.width = "100%"  # the sidebar's width; labels keep a fixed column
morning = w.Dropdown(options=list(STATES), index=2, description="Morning", disabled=True)
view = w.ToggleButtons(options=["The morning, hour by hour", "What the closure moved"], disabled=True,
                       style={"button_width": "14em"})
closes = w.HTML()
numbers, dials = w.HTML(), w.HTML(row(card("Time of day", "—"), card("Roads over capacity", "—")))
caption = w.HTML("The network, before anything moves. <b>Run</b> the model to watch commuters settle on it.")
curve = w.Image(format="png")
settled = w.VBox([curve, w.HTML(f"<small>The solver settling: each pass is commuters retrying routes and modes, "
                                f"not a day of Bayonne. The streets close after pass {WARMUP}.</small>")],
                 layout=w.Layout(display="none"))
table = w.HTML()
layer = road_layer()
city = Map(layer, view_state={"longitude": -1.478, "latitude": 43.4895, "zoom": 13.2}, height=520,
           basemap=MaplibreBasemap(style=CartoStyle.Positron), show_tooltip=True)
ran = {}  # the knobs the last run went with


def closure(_=None) -> None:
    areas, roads = SCENARIOS[scenario.value]
    closes.value = "Closed to cars: " + (", ".join(roads) or ", ".join(f"all of {a}" for a in areas)
                                         or "nothing, the city as it stands")


def play():
    """One run, pass by pass; the mornings wait for its end."""
    morning.disabled = view.disabled = True
    ran["knobs"] = {"mu": mu.value, "replan": replan.value, "peak_factor": peak.value}
    caption.value = "Building the graph: roads, zones and commuters. The first pass follows."
    return passes(scenario.value, period.value, sample.value, ran["knobs"])


def show(history: list, k: int) -> None:
    """Pass k: the peak's jam, the numbers against the last pass before the closure, the curve so far."""
    step = history[k]
    warm = step["phase"] == "warm-up"
    base = None if warm else next(p for p in reversed(history[:k]) if p["phase"] == "warm-up")
    over = jam(peak_at(step["flows"], 1.0, ran["knobs"]))
    dial("Pass", f"{step['day']} of {WARMUP + DAYS}",
         "routes settling on the census" if warm else "streets closed", over)
    caption.value = ("Commuters retry routes and modes on the last pass's travel times. The peak hour: volume "
                     "over capacity, light (free-flowing) to dark (saturated); width follows the flow.")
    figures(step, base)
    settling(history[: k + 1])


def mornings(_=None) -> None:
    """The chosen morning: its numbers, its streets, and the map, hour by hour or against before."""
    history, knobs = player.history, ran["knobs"]
    base = next(p for p in reversed(history) if p["phase"] == "warm-up")
    after = [p for p in history if p["phase"] != "warm-up"]
    now = {"before": base, "first": after[0], "last": after[-1]}[STATES[morning.value]]
    figures(now, None if now is base else base)
    streets(now, base)
    if view.value == "What the closure moved":
        over = moved(peak_at(now["flows"], 1.0, knobs), peak_at(base["flows"], 1.0, knobs)["flow"].to_numpy())
        dial("Time of day", "08:00", "the peak", over)
        caption.value = (f"<span style='color:{LESS}'>less traffic</span> · <span style='color:{MORE}'>more traffic"
                         "</span> than before the closure, at the peak; under 25 veh/h of change hidden")
        return
    caption.value = f"{morning.value}. Volume over capacity, light (free-flowing) to dark (saturated); width follows the flow."
    for i, (at, share) in enumerate(MORNING.items()):  # the settled peak, scaled back half hour by half hour
        if i:
            time.sleep(player.pace.value)
        over = jam(peak_at(now["flows"], share, knobs))
        dial("Time of day", at, f"{share:.0%} of the peak's cars", over)


def done(history: list) -> None:
    morning.disabled = view.disabled = False
    mornings()


player = Player(play, show, pace=1.0, unit="pass", done=done)
closure()
scenario.observe(closure, names="value")
morning.observe(player.errors.capture(clear_output=True)(mornings), names="value")
view.observe(player.errors.capture(clear_output=True)(mornings), names="value")

main = w.VBox([closes, w.HBox([morning, view]), numbers, dials, player.slider, city, caption, settled, table,
               player.errors], layout=w.Layout(flex="1"))
for child in main.children:  # a long column squeezes its children unless told not to
    if child is not city:  # the map has its own height, and "auto" lets it grow past it
        child.layout.flex = "0 0 auto"
w.HBox([
    w.VBox([scenario, period,
            w.Accordion([w.VBox([mu, replan, peak, sample])], titles=("Behaviour and calibration",)),
            player.pace, player.button], layout=w.Layout(width="400px")),
    main,
])

In [ ]:
what = Markdown(f"""
- **Agents**: a sample (the *Agent sample* control) of the {parts["flows"]["workers"].sum():,.0f} commuters living or
  working in the 13 communes of the Bayonne–Anglet–Biarritz core (INSEE census 2022, flows by mode, via data.gouv.fr),
  in {parts["zones"].height:,} zones.
- **Network**: {links.height:,} directed road links from OpenStreetMap — every street of the core, the main roads
  from Hendaye to Dax — with speed limits, lanes and bus lanes as mapped.
- **How a morning is computed**: commuters retry routes and modes until the assignment stops moving — nested logit
  on the previous pass's travel times (Horowitz 1984; Cascetta 1989), mode shares pivoting on the census
  (Koppelman 1983), links congesting by BPR. Those passes are how the equilibrium is *solved*, not days of
  Bayonne; only three mornings mean anything, and they are the ones you can pick.
- **The morning replay** scales that settled peak by an assumed profile ({", ".join(f"{k} {v:.0%}" for k, v in MORNING.items())})
  and re-applies the same delay curve: flows fall linearly, delay does not. No queue carries from one half hour to
  the next, and an hourly count profile is the first thing that should replace it.
- **Not modelled**: departure-time shifts, destination changes and trips given up (so traffic "evaporation" is only
  mode shift), queues spilling back, parking capacity, through traffic on the A63 that neither starts nor ends in
  the area, tourists, deliveries, residents' permits.
- **Noise**: at a 10% sample, mode-share changes under ~0.3 points are seed noise, not signal.
- **Periods** scale commuting volume by assumption ({", ".join(f"{k}: {v:.0%}" for k, v in PERIODS.items())}) until
  seasonal counts calibrate them; the fêtes de Bayonne are the natural experiment that can calibrate the rest.
""")
about = w.Output()
with about:
    display(what)
w.Accordion([about], titles=("What this model is, and is not",))